# Sports Teammate Finding & Court Booking Survey Analysis

Notebook này phân tích `response_details.csv`, reverse-engineer các giả thuyết nghiên cứu từ bảng khảo sát hiện có, và tạo báo cáo business + methodology bằng tiếng Việt. File raw data không bị chỉnh sửa.

## Workflow

1. Inspect raw data and missingness.
2. Clean categories and parse multiple-choice answers.
3. Create hypothesis evidence, segment comparisons, WTP analysis, feature prioritization, and hidden insights.
4. Save cleaned data, CSV tables, charts, and Vietnamese markdown reports.
5. Explicitly report methodology/compliance gaps instead of fabricating unsupported survey-process details.

## 1. Setup & Configuration

Initialize paths, create report directories, configure plotting settings, and load the raw survey dataset.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
RAW_PATH = ROOT / "response_details.csv"
DATA_DIR = ROOT / "data"
REPORTS_DIR = ROOT / "reports"
CHARTS_DIR = REPORTS_DIR / "charts"
TABLES_DIR = REPORTS_DIR / "tables"
for d in [DATA_DIR, REPORTS_DIR, CHARTS_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["font.family"] = "DejaVu Sans"

raw = pd.read_csv(RAW_PATH)
N = len(raw)
original_columns = list(raw.columns)

column_map = {
    "UID": "uid",
    "2. Độ tuổi của bạn là bao nhiêu?": "age",
    "3. Giới tính của bạn là gì?": "gender",
    "4. Nghề nghiệp hiện tại của bạn là gì?": "occupation",
    "Thu nhập của bạn khoảng bao nhiêu:": "income",
    "6. Bạn hiện có chơi hoặc quan tâm đến các hoạt động thể thao không?": "sport_interest",
    "7. Bạn đang chơi hoặc muốn chơi những môn thể thao nào?": "sports",
    "8. Tần suất bạn muốn chơi thể thao lý tưởng là bao nhiêu?": "ideal_frequency",
    "9. Bạn đã từng muốn chơi thể thao nhưng không đủ người chơi cùng chưa?": "teammate_shortage",
    "10. Khó khăn lớn nhất của bạn khi tìm người chơi cùng là gì?": "teammate_pain_points",
    "11. Nếu có hệ thống ghép người chơi theo khu vực gần đúng, trình độ và lịch rảnh, bạn có sẵn sàng chơi với người mới không?": "new_player_openness",
    "12. Khi ghép với người chơi mới, yếu tố nào khiến bạn tin tưởng hơn?": "trust_factors",
    "13. Hiện tại bạn thường dùng cách nào để tìm người chơi thể thao cùng?": "current_finding_methods",
    "14. Điểm bất tiện lớn nhất khi tìm người chơi bằng các cách hiện tại là gì?": "current_finding_pains",
    "15. Bạn có thường cần đặt sân hoặc tìm địa điểm khi chơi thể thao không?": "booking_need",
    "16. Hiện tại bạn thường đặt sân hoặc tìm địa điểm chơi bằng cách nào?": "current_booking_methods",
    "17. Điểm bất tiện lớn nhất khi đặt sân/tìm địa điểm hiện nay là gì?": "booking_pains",
    "18. Nếu có một ứng dụng tổng hợp cả tìm người chơi, đặt sân/địa điểm và cộng đồng thể thao, điều gì sẽ khiến bạn chuyển sang dùng?": "switching_triggers",
    "19. Nếu ứng dụng giúp bạn tìm người chơi và địa điểm dễ hơn, bạn nghĩ mình sẽ chơi thể thao thường xuyên hơn không?": "expected_frequency_lift",
    "20. Bạn có sẵn sàng trả phí nếu ứng dụng thật sự giúp bạn tìm người chơi hoặc đặt sân thuận tiện hơn không?": "willingness_to_pay",
    "21. Bạn sẵn sàng trả phí cho tính năng nào?": "paid_features",
    "Điều gì có thể khiến bạn quan tâm hơn đến việc chơi thể thao?": "open_motivation",
}



## 2. Data Cleaning & Standardization

Rename columns to user-friendly names, trim whitespaces, parse multi-select responses into arrays, and compute behavioral and eligibility scores (e.g. teammate shortage score, early adopter candidates).


In [2]:
clean = raw.rename(columns=column_map).copy()
for col in clean.select_dtypes(include=["object", "string"]).columns:
    clean[col] = clean[col].fillna("").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

multi_cols = [
    "sports", "teammate_pain_points", "trust_factors", "current_finding_methods",
    "current_finding_pains", "current_booking_methods", "booking_pains",
    "switching_triggers", "paid_features", "open_motivation"
]

def split_multi(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [x.strip() for x in str(value).split(",") if x.strip()]

for col in multi_cols:
    clean[col + "_list"] = clean[col].apply(split_multi)

clean["new_player_openness_num"] = pd.to_numeric(clean["new_player_openness"].replace("", np.nan), errors="coerce")
clean["high_openness_to_new_players"] = clean["new_player_openness_num"] >= 4

shortage_score = {"Chưa bao giờ": 0, "Hiếm khi": 1, "Thỉnh thoảng": 2, "Rất thường xuyên": 3}
frequency_score = {"Chưa rõ": 0, "Khi có người rủ": 1, "1–3 lần/tháng": 1, "1–2 lần/tuần": 2, "3 lần/tuần trở lên": 3}
booking_score = {"Không bao giờ": 0, "Hiếm khi": 1, "Thỉnh thoảng": 2, "Thường xuyên": 3}
wtp_score = {"Không": 0, "Có thể, nếu giá hợp lý": 1, "Có thể, nếu app thật sự có nhiều người dùng": 1, "Có": 2}
lift_score = {"Không": 0, "Không chắc": 0, "Có thể": 1, "Có, chắc chắn": 2}
interest_score = {"Không quan tâm": 0, "Trước đây có chơi nhưng hiện tại ít chơi": 1, "Chưa chơi thường xuyên nhưng muốn bắt đầu": 1, "Có, thỉnh thoảng": 2, "Có, thường xuyên": 3}

clean["teammate_shortage_score"] = clean["teammate_shortage"].map(shortage_score)
clean["ideal_frequency_score"] = clean["ideal_frequency"].map(frequency_score)
clean["booking_need_score"] = clean["booking_need"].map(booking_score)
clean["wtp_score"] = clean["willingness_to_pay"].map(wtp_score)
clean["expected_frequency_lift_score"] = clean["expected_frequency_lift"].map(lift_score)
clean["sport_interest_score"] = clean["sport_interest"].map(interest_score)
clean["pain_count_teammate"] = clean["teammate_pain_points_list"].apply(len)
clean["pain_count_booking"] = clean["booking_pains_list"].apply(len)
clean["early_adopter_candidate"] = clean["sport_interest_score"].fillna(0).ge(2) & clean["teammate_shortage_score"].fillna(0).ge(2) & clean["new_player_openness_num"].fillna(0).ge(4) & clean["expected_frequency_lift_score"].fillna(0).ge(1)
clean["student_young_segment"] = clean["age"].eq("18–22") & clean["occupation"].eq("Học sinh/Sinh viên")
clean["has_wtp_signal"] = clean["wtp_score"].fillna(0).gt(0)
clean["strong_wtp"] = clean["willingness_to_pay"].eq("Có")

clean.drop(columns=[c for c in clean.columns if c.endswith("_list")]).to_csv(DATA_DIR / "cleaned_survey.csv", index=False, encoding="utf-8-sig")



## 3. Metadata & Log Generation

Generate the data dictionary, check for missing values, and output a structured cleaning log.


In [3]:
data_dictionary = pd.DataFrame([
    {
        "original_column": orig,
        "clean_name": column_map.get(orig, orig),
        "usage": (
            "identifier" if orig == "UID" else
            "demographic" if column_map.get(orig) in ["age", "gender", "occupation", "income"] else
            "behavior" if column_map.get(orig) in ["sport_interest", "sports", "ideal_frequency"] else
            "problem_evidence" if column_map.get(orig) in ["teammate_shortage", "teammate_pain_points", "current_finding_methods", "current_finding_pains", "booking_need", "current_booking_methods", "booking_pains"] else
            "solution_demand" if column_map.get(orig) in ["new_player_openness", "trust_factors", "switching_triggers", "expected_frequency_lift", "willingness_to_pay", "paid_features"] else
            "qualitative"
        ),
    }
    for orig in original_columns
])
data_dictionary.to_csv(TABLES_DIR / "data_dictionary.csv", index=False, encoding="utf-8-sig")

missing = pd.DataFrame({"column": raw.columns, "missing_count": [raw[c].isna().sum() + raw[c].astype(str).str.strip().eq("").sum() for c in raw.columns]})
missing["missing_pct"] = (missing["missing_count"] / N * 100).round(1)
missing.to_csv(TABLES_DIR / "missing_values.csv", index=False, encoding="utf-8-sig")

pd.DataFrame([
    {"step": "rename_columns", "detail": "Mapped original Vietnamese question text to stable snake_case names."},
    {"step": "trim_text", "detail": "Trimmed whitespace and collapsed repeated spaces in text fields."},
    {"step": "parse_multi_select", "detail": "Split comma-separated multi-select answers into option lists for counting."},
    {"step": "score_fields", "detail": "Created ordinal scores for pain frequency, booking need, WTP, frequency lift, and openness."},
    {"step": "preserve_raw", "detail": "Raw response_details.csv was read-only and not modified."},
]).to_csv(TABLES_DIR / "cleaning_log.csv", index=False, encoding="utf-8-sig")



## 4. Analysis Helper Functions & Descriptive Statistics

Define helper functions to summarize single and multi-select survey questions and write the basic tables to CSVs.


In [4]:
def single_summary(df, col):
    valid = df[col].replace("", np.nan).dropna()
    out = valid.value_counts().rename_axis("option").reset_index(name="count")
    out["pct_valid"] = (out["count"] / max(len(valid), 1) * 100).round(1)
    out["pct_total"] = (out["count"] / N * 100).round(1)
    return out

def multi_summary(df, list_col, label):
    exploded = df[[list_col]].explode(list_col)
    exploded = exploded[exploded[list_col].notna() & exploded[list_col].astype(str).str.strip().ne("")]
    out = exploded[list_col].value_counts().rename_axis("option").reset_index(name="count")
    respondent_count = df[list_col].apply(lambda x: len(x) > 0).sum()
    out["pct_valid_respondents"] = (out["count"] / max(respondent_count, 1) * 100).round(1)
    out["pct_total_respondents"] = (out["count"] / N * 100).round(1)
    out.insert(0, "question", label)
    return out

def save_bar(table, x_col, y_col, title, filename, top=10):
    t = table.head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(t))))
    ax.barh(t[x_col], t[y_col], color="#2F6F73")
    ax.set_title(title)
    ax.set_xlabel("So phan hoi")
    for i, v in enumerate(t[y_col]):
        ax.text(float(v) + 0.5, i, str(round(float(v), 2)), va="center", fontsize=9)
    fig.tight_layout()
    fig.savefig(CHARTS_DIR / filename, dpi=160, bbox_inches="tight")
    plt.close(fig)

single_cols = ["age", "gender", "occupation", "income", "sport_interest", "ideal_frequency", "teammate_shortage", "new_player_openness", "booking_need", "expected_frequency_lift", "willingness_to_pay"]
question_summaries = pd.concat([single_summary(clean, col).assign(field=col)[["field", "option", "count", "pct_valid", "pct_total"]] for col in single_cols], ignore_index=True)
question_summaries.to_csv(TABLES_DIR / "question_summaries.csv", index=False, encoding="utf-8-sig")

sports_tbl = multi_summary(clean, "sports_list", "Sports currently played or desired")
teammate_pain_tbl = multi_summary(clean, "teammate_pain_points_list", "Teammate finding pain points")
trust_tbl = multi_summary(clean, "trust_factors_list", "Trust factors for new players")
finding_methods_tbl = multi_summary(clean, "current_finding_methods_list", "Current teammate finding methods")
finding_pains_tbl = multi_summary(clean, "current_finding_pains_list", "Pains in current teammate finding methods")
booking_methods_tbl = multi_summary(clean, "current_booking_methods_list", "Current booking/location methods")
booking_pains_tbl = multi_summary(clean, "booking_pains_list", "Booking/location pain points")
switching_tbl = multi_summary(clean, "switching_triggers_list", "Switching triggers")
paid_features_tbl = multi_summary(clean, "paid_features_list", "Paid feature preferences")
open_motivation_tbl = multi_summary(clean, "open_motivation_list", "Open-ended motivations")

for name, tbl in [("sports.csv", sports_tbl), ("pain_points_teammate.csv", teammate_pain_tbl), ("trust_factors.csv", trust_tbl), ("alternatives.csv", finding_methods_tbl), ("current_finding_pains.csv", finding_pains_tbl), ("booking_methods.csv", booking_methods_tbl), ("booking_pains.csv", booking_pains_tbl), ("switching_triggers.csv", switching_tbl), ("wtp_paid_features.csv", paid_features_tbl), ("open_motivation_themes.csv", open_motivation_tbl)]:
    tbl.to_csv(TABLES_DIR / name, index=False, encoding="utf-8-sig")

pain_points = pd.concat([teammate_pain_tbl.assign(area="Tìm người chơi"), finding_pains_tbl.assign(area="Cách tìm người hiện tại"), booking_pains_tbl.assign(area="Đặt sân/tìm địa điểm")], ignore_index=True)
pain_points.to_csv(TABLES_DIR / "pain_points.csv", index=False, encoding="utf-8-sig")



## 5. Segment & Cohort Analysis

Define segment cohorts and analyze conditional user pain, interest, and Willingness-To-Pay (WTP) comparisons.


In [5]:
def segment_metrics(mask, name):
    d = clean[mask].copy()
    n = len(d)
    return {
        "segment": name,
        "n": n,
        "pct_sample": round(n / N * 100, 1),
        "avg_teammate_shortage_score": round(d["teammate_shortage_score"].mean(), 2),
        "avg_openness_to_new_players": round(d["new_player_openness_num"].mean(), 2),
        "expected_lift_yes_or_maybe_pct": round(d["expected_frequency_lift_score"].fillna(0).gt(0).mean() * 100, 1),
        "wtp_any_pct": round(d["has_wtp_signal"].mean() * 100, 1),
        "strong_wtp_pct": round(d["strong_wtp"].mean() * 100, 1),
        "booking_need_often_or_sometimes_pct": round(d["booking_need_score"].fillna(0).ge(2).mean() * 100, 1),
        "early_adopter_candidate_pct": round(d["early_adopter_candidate"].mean() * 100, 1),
    }

segment_tbl = pd.DataFrame([
    segment_metrics(clean.index == clean.index, "Toàn mẫu"),
    segment_metrics(clean["student_young_segment"], "18-22 và Học sinh/Sinh viên"),
    segment_metrics(clean["sport_interest_score"].fillna(0).ge(2), "Đang chơi/quan tâm thể thao"),
    segment_metrics(clean["teammate_shortage_score"].fillna(0).ge(2), "Thiếu người chơi thỉnh thoảng/rất thường xuyên"),
    segment_metrics(clean["new_player_openness_num"].fillna(0).ge(4), "Sẵn sàng chơi với người mới (4-5)"),
    segment_metrics(clean["booking_need_score"].fillna(0).ge(2), "Cần đặt sân/tìm địa điểm thỉnh thoảng/thường xuyên"),
    segment_metrics(clean["has_wtp_signal"], "Có tín hiệu trả phí"),
    segment_metrics(clean["early_adopter_candidate"], "Ứng viên early adopter"),
])
segment_tbl.to_csv(TABLES_DIR / "segment_comparison.csv", index=False, encoding="utf-8-sig")



## 6. MVP Feature Prioritization & WTP Signals

Extract and rank potential product features based on a weighted priority score combining user pain points, adoption triggers, monetization indicators, and technical feasibility.


In [6]:
wtp_tbl = single_summary(clean, "willingness_to_pay")
wtp_tbl.to_csv(TABLES_DIR / "wtp_summary.csv", index=False, encoding="utf-8-sig")

switch_counts = dict(zip(switching_tbl["option"], switching_tbl["count"]))
trust_counts = dict(zip(trust_tbl["option"], trust_tbl["count"]))
paid_counts = dict(zip(paid_features_tbl["option"], paid_features_tbl["count"]))
pain_counts = dict(zip(pain_points["option"], pain_points["count"]))
features = [
    {"feature": "Ghép người chơi theo lịch rảnh", "evidence_sources": "switching_triggers + teammate pain", "pain_score": pain_counts.get("Khó tìm người cùng lịch rảnh", 0), "adoption_score": switch_counts.get("Ghép người chơi theo lịch rảnh", 0), "wtp_score": paid_counts.get("Phí tham gia trận/nhóm chơi", 0), "feasibility_score": 4},
    {"feature": "Ghép người chơi theo trình độ", "evidence_sources": "switching_triggers + trust factors + teammate pain", "pain_score": pain_counts.get("Khó tìm người cùng trình độ", 0) + pain_counts.get("Không biết trình độ người chơi", 0), "adoption_score": switch_counts.get("Ghép người chơi theo trình độ", 0) + trust_counts.get("Biết trước trình độ chơi", 0), "wtp_score": paid_counts.get("Gói premium để được ghép người phù hợp hơn", 0), "feasibility_score": 3},
    {"feature": "Lịch trống và giá sân minh bạch", "evidence_sources": "booking_pains + switching_triggers + paid_features", "pain_score": pain_counts.get("Không biết sân/địa điểm nào còn trống", 0) + pain_counts.get("Không rõ giá", 0), "adoption_score": switch_counts.get("Biết trước lịch trống và giá sân", 0), "wtp_score": paid_counts.get("Phí đặt sân/địa điểm tiện lợi", 0), "feasibility_score": 3},
    {"feature": "Hồ sơ, đánh giá và xác minh người chơi", "evidence_sources": "trust_factors + switching_triggers + current finding pains", "pain_score": pain_counts.get("Ngại chơi với người lạ vì thiếu thông tin", 0) + pain_counts.get("Không có đánh giá/độ uy tín của người chơi", 0), "adoption_score": trust_counts.get("Hồ sơ người chơi rõ ràng", 0) + trust_counts.get("Có đánh giá từ người chơi khác", 0) + trust_counts.get("Có xác minh tài khoản/số điện thoại", 0), "wtp_score": 0, "feasibility_score": 3},
    {"feature": "Nhóm chat trước buổi chơi", "evidence_sources": "trust_factors", "pain_score": pain_counts.get("Ngại chơi với người lạ vì thiếu thông tin", 0), "adoption_score": trust_counts.get("Có nhóm chat trước buổi chơi", 0), "wtp_score": 0, "feasibility_score": 5},
    {"feature": "Nhiều người chơi thật / bootstrap cộng đồng", "evidence_sources": "switching_triggers + WTP conditional on user density", "pain_score": pain_counts.get("Không đủ số lượng người để chơi", 0), "adoption_score": switch_counts.get("Có nhiều người chơi thật", 0), "wtp_score": clean["willingness_to_pay"].eq("Có thể, nếu app thật sự có nhiều người dùng").sum(), "feasibility_score": 2},
    {"feature": "Đối tác sân/địa điểm thể thao", "evidence_sources": "switching_triggers + booking methods/pains", "pain_score": pain_counts.get("Phải gọi/nhắn nhiều nơi", 0) + pain_counts.get("Không biết sân/địa điểm nào còn trống", 0), "adoption_score": switch_counts.get("Có nhiều sân/địa điểm thể thao đối tác", 0), "wtp_score": paid_counts.get("Phí đặt sân/địa điểm tiện lợi", 0), "feasibility_score": 2},
    {"feature": "Giải/sự kiện thể thao", "evidence_sources": "paid_features", "pain_score": 0, "adoption_score": 0, "wtp_score": paid_counts.get("Phí tham gia giải hoặc sự kiện thể thao", 0), "feasibility_score": 2},
]
feature_tbl = pd.DataFrame(features)
for col in ["pain_score", "adoption_score", "wtp_score", "feasibility_score"]:
    feature_tbl[col + "_norm"] = feature_tbl[col] / (feature_tbl[col].max() or 1)
feature_tbl["weighted_score"] = (0.35 * feature_tbl["pain_score_norm"] + 0.30 * feature_tbl["adoption_score_norm"] + 0.20 * feature_tbl["wtp_score_norm"] + 0.15 * feature_tbl["feasibility_score_norm"]).round(3)
feature_tbl["priority"] = pd.cut(feature_tbl["weighted_score"], bins=[-0.01, 0.33, 0.62, 1.01], labels=["Ignore/Defer", "Nice-to-have", "Must-have"])
feature_tbl = feature_tbl.sort_values("weighted_score", ascending=False)
feature_tbl.to_csv(TABLES_DIR / "feature_priority.csv", index=False, encoding="utf-8-sig")



## 7. Hypotheses Evaluation & Hidden Insights

Evaluate research hypotheses (H1-H6) against descriptive data evidence and format the core hidden business insights.


In [7]:
metrics = {
    "shortage_sometimes_or_often": int(clean["teammate_shortage_score"].fillna(0).ge(2).sum()),
    "openness_4_5": int(clean["new_player_openness_num"].fillna(0).ge(4).sum()),
    "frequency_lift_yes_or_maybe": int(clean["expected_frequency_lift_score"].fillna(0).gt(0).sum()),
    "wtp_any": int(clean["has_wtp_signal"].sum()),
    "wtp_strong": int(clean["strong_wtp"].sum()),
    "booking_need_sometimes_or_often": int(clean["booking_need_score"].fillna(0).ge(2).sum()),
    "early_adopter_candidates": int(clean["early_adopter_candidate"].sum()),
}

hypotheses = pd.DataFrame([
    {
        "hypothesis": "H1: Tồn tại nhu cầu đối với việc tìm bạn chơi thể thao và đặt sân/địa điểm tiện lợi.",
        "evidence": f"{metrics['shortage_sometimes_or_often']}/{N} người báo cáo thiếu bạn chơi ít nhất thỉnh thoảng; {metrics['frequency_lift_yes_or_maybe']}/{N} người cho rằng ứng dụng có thể giúp họ chơi thể thao thường xuyên hơn.",
        "assessment": "Được ủng hộ, nhưng mẫu lệch nhiều về sinh viên và không mang tính đại diện toàn bộ."
    },
    {
        "hypothesis": "H2: Nhóm học sinh/sinh viên trẻ tuổi là phân khúc người dùng sớm (early adopter) triển vọng nhất.",
        "evidence": f"{int(clean['student_young_segment'].sum())}/{N} là sinh viên/học sinh từ 18-22 tuổi; so sánh phân khúc cho thấy mức độ bất tiện, sẵn sàng mở lòng và ý định trả phí của nhóm này cao hơn mẫu chung.",
        "assessment": "Được ủng hộ như một phân khúc thực tế để launch (beachhead segment), không phải bằng chứng cho toàn bộ thị trường TP.HCM."
    },
    {
        "hypothesis": "H3: Các cơ chế xây dựng lòng tin là yêu cầu bắt buộc để người dùng chấp nhận chơi với người lạ.",
        "evidence": f"{metrics['openness_4_5']}/{N} người đánh giá mức độ sẵn sàng chơi với người mới ở mức cao (4-5); bảng trust-factor cho thấy hồ sơ, đánh giá, xác minh thông tin và nhóm chat là các yêu cầu cốt lõi.",
        "assessment": "Được ủng hộ; việc ghép trận mà không có cơ chế tin cậy sẽ không giải quyết được rào cản hành vi."
    },
    {
        "hypothesis": "H4: Tiện ích đặt sân và tìm địa điểm là wedge (bước đệm) monetization (tạo doanh thu) mạnh nhất.",
        "evidence": f"{metrics['booking_need_sometimes_or_often']}/{N} người cần đặt sân hoặc tìm địa điểm ít nhất thỉnh thoảng; bảng tính năng trả phí xếp phí tiện ích đặt sân cao hơn nhiều so với các loại phí ghép người/sự kiện.",
        "assessment": "Được ủng hộ về mặt định hướng; mức giá cụ thể chưa thể ước lượng vì bảng khảo sát không hỏi các mức giá."
    },
    {
        "hypothesis": "H5: Các giải pháp thay thế hiện tại (Facebook, Zalo, bạn bè) quá phân mảnh và gây bất tiện.",
        "evidence": "Các bảng phương pháp hiện tại và pain points cho thấy người dùng phải rủ thủ công qua Zalo/Facebook, nhắn tin nhiều nơi, dễ bị hủy kèo và không biết lịch trống của sân.",
        "assessment": "Được ủng hộ; cơ hội mở ra nếu sản phẩm giải quyết được vấn đề kết nối cung-cầu (liquidity) và đối tác sân."
    },
    {
        "hypothesis": "H6: Khảo sát ý tưởng (validation) hiện tại vẫn chưa hoàn tất do các hạn chế về phương pháp và tín hiệu WTP.",
        "evidence": f"Chỉ có {metrics['wtp_strong']}/{N} trả lời chắc chắn 'Có' cho việc trả phí; các thông tin về recruitment, response rate, consent và việc tách file email/PII không thể kiểm chứng trực tiếp từ file CSV phân tích.",
        "assessment": "Được ủng hộ; các bước validation tiếp theo cần đo lường tỷ lệ chuyển đổi đặt sân thực tế và kiểm chứng mức giá."
    }
])
hypotheses.to_csv(TABLES_DIR / "hypotheses.csv", index=False, encoding="utf-8-sig")

hidden_insights = pd.DataFrame([
    {"hidden_insight": "Nhu cầu có dấu hiệu thật, nhưng chưa phải nhu cầu trả phí mạnh.", "evidence": f"{metrics['frequency_lift_yes_or_maybe']}/{N} kỳ vọng chơi thường xuyên hơn, nhưng chỉ {metrics['wtp_strong']}/{N} trả lời chắc chắn sẵn sàng trả phí.", "business_meaning": "Không nên bắt đầu bằng subscription rộng; cần gắn phí với giao dịch có giá trị rõ như đặt sân hoặc tham gia trận.", "action": "Ưu tiên phí tiện ích/hoa hồng đặt sân và kiểm chứng willingness-to-pay bằng landing page hoặc concierge MVP."},
    {"hidden_insight": "App chỉ có giá trị nếu đạt mật độ người chơi thật.", "evidence": f"{metrics['wtp_any'] - metrics['wtp_strong']}/{N} có WTP có điều kiện hoặc không chắc chắn; switching trigger 'Có nhiều người chơi thật' xuất hiện nổi bật.", "business_meaning": "Rủi ro chicken-and-egg lớn hơn rủi ro tính năng.", "action": "Launch theo cụm nhỏ: một vài trường/khu vực/môn thể thao trước khi mở rộng toàn thành phố."},
    {"hidden_insight": "Vấn đề tin tưởng là rào cản nền tảng khi ghép người lạ.", "evidence": "Trust-factor table cho thấy hồ sơ, đánh giá, xác minh, trình độ và nhóm chat đều xuất hiện lặp lại.", "business_meaning": "Ghép lịch và trình độ chưa đủ; thiếu trust sẽ làm giảm conversion sang buổi chơi thật.", "action": "MVP phải có hồ sơ cơ bản, review sau buổi chơi, xác minh số điện thoại và chat trước buổi chơi."},
    {"hidden_insight": "Đặt sân là pain point có thể kiếm tiền hơn cộng đồng thuần túy.", "evidence": f"{metrics['booking_need_sometimes_or_often']}/{N} cần đặt sân/tìm địa điểm ít nhất thỉnh thoảng; paid-feature table có tín hiệu phí đặt sân/địa điểm tiện lợi cao.", "business_meaning": "Booking tạo giao dịch rõ ràng và dễ justify fee hơn tính năng cộng đồng.", "action": "Ưu tiên dữ liệu sân, lịch trống, giá, đánh giá sân và quy trình đặt/giữ chỗ."},
    {"hidden_insight": "Mẫu khảo sát rất lệch về sinh viên 18-22.", "evidence": f"{int(clean['age'].eq('18–22').sum())}/{N} ở nhóm 18-22 và {int(clean['occupation'].eq('Học sinh/Sinh viên').sum())}/{N} là học sinh/sinh viên.", "business_meaning": "Kết quả phù hợp để chọn beachhead segment, không đủ để khẳng định đại diện toàn bộ người chơi thể thao tại TP.HCM.", "action": "Báo cáo demand là promising trong nhóm trẻ/sinh viên; validation tiếp theo cần mẫu người đi làm và người đặt sân thường xuyên."}
])
hidden_insights.to_csv(TABLES_DIR / "hidden_insights.csv", index=False, encoding="utf-8-sig")


## 8. Data Visualizations

Generate and save key demographic, pain point, and priority visualization charts as PNG files.


In [8]:
save_bar(single_summary(clean, "age"), "option", "count", "Phan bo do tuoi", "age_distribution.png")
save_bar(single_summary(clean, "occupation"), "option", "count", "Phan bo nghe nghiep", "occupation_distribution.png")
save_bar(single_summary(clean, "teammate_shortage"), "option", "count", "Tan suat thieu nguoi choi cung", "teammate_shortage.png")
save_bar(teammate_pain_tbl, "option", "count", "Pain points khi tim nguoi choi", "teammate_pain_points.png")
save_bar(booking_pains_tbl, "option", "count", "Pain points khi dat san/tim dia diem", "booking_pains.png")
save_bar(switching_tbl, "option", "count", "Yeu to khien nguoi dung chuyen sang app", "switching_triggers.png")
save_bar(feature_tbl, "feature", "weighted_score", "Uu tien tinh nang MVP", "feature_priority.png", top=8)
save_bar(wtp_tbl, "option", "count", "Muc san sang tra phi", "wtp_summary.png")



## 9. Markdown Report Generation

Compile all descriptive metrics, hypothesis assessments, and segment stats into the final Vietnamese business insights report, methodology compliance report, and project README.


In [9]:
def top_options(tbl, n=5):
    return "; ".join([f"{r.option}: {int(r.count)} ({r.pct_total_respondents}% toàn mẫu)" for r in tbl.head(n).itertuples()])

def single_options(col, n=5):
    tbl = single_summary(clean, col).head(n)
    return "; ".join([f"{r.option}: {int(r.count)} ({r.pct_total}% toàn mẫu)" for r in tbl.itertuples()])

student_segment = segment_tbl[segment_tbl["segment"].eq("18-22 và Học sinh/Sinh viên")].iloc[0]
final_decision = "Có tín hiệu demand đủ để tiếp tục MVP hẹp, nhưng chưa đủ để kết luận thị trường TP.HCM rộng lớn đã được xác thực."
feature_rows = "\n".join([f"| {r.feature} | {r.priority} | {r.weighted_score} | {r.evidence_sources} |" for r in feature_tbl.itertuples()])
hidden_rows = "\n".join([f"### {i+1}. {r.hidden_insight}\n- Evidence: {r.evidence}\n- Business Meaning: {r.business_meaning}\n- Action: {r.action}" for i, r in hidden_insights.iterrows()])
hidden_rows_h4 = "\n".join([f"#### {i+1}. {r.hidden_insight}\n- Evidence: {r.evidence}\n- Business Meaning: {r.business_meaning}\n- Action: {r.action}" for i, r in hidden_insights.iterrows()])
hypothesis_rows = "\n".join([f"- {r.hypothesis} {r.assessment}" for r in hypotheses.itertuples()])

insights_md = f"""# Báo cáo phân tích khảo sát startup thể thao

## Executive Summary
- Bộ dữ liệu có **{N} phản hồi** và **{len(original_columns)} cột**. File gốc `response_details.csv` được giữ nguyên; dữ liệu đã làm sạch được lưu tại `data/cleaned_survey.csv`.
- Kết luận chính: **{final_decision}**
- Lý do ủng hộ demand: **{metrics['shortage_sometimes_or_often']}/{N}** người từng thiếu người chơi cùng ít nhất thỉnh thoảng; **{metrics['frequency_lift_yes_or_maybe']}/{N}** cho rằng app có thể hoặc chắc chắn giúp họ chơi thường xuyên hơn; **{metrics['booking_need_sometimes_or_often']}/{N}** cần đặt sân/tìm địa điểm ít nhất thỉnh thoảng.
- Điểm yếu: mẫu lệch mạnh về nhóm 18-22 và học sinh/sinh viên; WTP chắc chắn chỉ **{metrics['wtp_strong']}/{N}**; chưa có dữ liệu giá cụ thể hoặc thông tin tính toán response rate.

## Dataset Overview
- Kích thước: {N} dòng x {len(original_columns)} cột.
- Thiếu dữ liệu đáng chú ý: câu hỏi thể thao và demand có 10 dòng trống; nhóm đặt sân có 21 dòng trống; câu hỏi mở `open_motivation` rất thưa với 141 dòng trống.
- Bảng hỗ trợ: `reports/tables/data_dictionary.csv`, `reports/tables/missing_values.csv`, `reports/tables/cleaning_log.csv`.

## Key Findings
### Nhân khẩu học
- Tuổi: {single_options('age', 4)}.
- Nghề nghiệp: {single_options('occupation', 5)}.
- Giới tính: {single_options('gender', 4)}.
- Diễn giải: mẫu phù hợp để hiểu nhóm sinh viên/trẻ tại TP.HCM, nhưng không đại diện cho toàn bộ người chơi thể thao hoặc người đi làm.

![Phân bố độ tuổi](charts/age_distribution.png)
![Phân bố nghề nghiệp](charts/occupation_distribution.png)

### Hành vi thể thao
- Mức quan tâm/chơi thể thao: {single_options('sport_interest', 5)}.
- Tần suất lý tưởng: {single_options('ideal_frequency', 5)}.
- Môn thể thao nổi bật: {top_options(sports_tbl, 6)}.

### Problem & Solution Fit
- Thiếu người chơi: {single_options('teammate_shortage', 4)}.

![Tần suất thiếu người chơi cùng](charts/teammate_shortage.png)
- Pain point tìm người chơi: {top_options(teammate_pain_tbl, 6)}.

![Khó khăn khi tìm người chơi cùng](charts/teammate_pain_points.png)
- Pain point của cách tìm hiện tại: {top_options(finding_pains_tbl, 6)}.
- Nhu cầu đặt sân/tìm địa điểm: {single_options('booking_need', 4)}.
- Pain point đặt sân/tìm địa điểm: {top_options(booking_pains_tbl, 6)}.

![Khó khăn khi đặt sân/địa điểm](charts/booking_pains.png)

## Hidden Insights
{hidden_rows}

## Target Segments
- Primary early adopter đề xuất: **18-22 và học sinh/sinh viên đang có pain thiếu người chơi, sẵn sàng gặp người mới, và có kỳ vọng app giúp chơi thường xuyên hơn**.
- Segment early adopter theo rule định lượng có **{metrics['early_adopter_candidates']}/{N}** người.
- Segment sinh viên 18-22 chiếm **{int(clean['student_young_segment'].sum())}/{N}** mẫu; trong bảng segment, nhóm này có WTP bất kỳ **{student_segment['wtp_any_pct']}%** và ứng viên early adopter **{student_segment['early_adopter_candidate_pct']}%**.
- Secondary segment: người có nhu cầu đặt sân/tìm địa điểm thỉnh thoảng hoặc thường xuyên, vì có liên hệ trực tiếp với booking convenience và monetization.

## Main Pain Points
- Tìm người chơi: bạn bè/người quen không rảnh, không đủ số lượng, khó khớp lịch, khó khớp trình độ, thiếu thông tin/uy tín khi gặp người lạ.
- Đặt sân/tìm địa điểm: không biết lịch trống, phải gọi/nhắn nhiều nơi, không rõ giá, không rõ chất lượng sân, dễ hết chỗ/trùng lịch.
- Ý nghĩa: app nên giải quyết coordination + trust + booking transparency trước khi mở rộng sang social/community features.

## Alternatives
- Cách tìm người hiện tại: {top_options(finding_methods_tbl, 6)}.
- Cách đặt sân hiện tại: {top_options(booking_methods_tbl, 6)}.
- Diễn giải: người dùng đang dùng giải pháp thủ công và phân mảnh, chủ yếu qua bạn bè, nhóm chat, Facebook/Zalo, gọi điện hoặc nhờ người khác đặt.

## MVP Feature Table
| Feature | Priority | Score | Evidence |
|---|---:|---:|---|
{feature_rows}

![Độ ưu tiên các tính năng MVP](charts/feature_priority.png)

## Monetization Signal
- WTP: {single_options('willingness_to_pay', 4)}.

![Mức độ sẵn sàng trả phí](charts/wtp_summary.png)
- Tính năng trả phí: {top_options(paid_features_tbl, 6)}.
- Khuyến nghị business model: bắt đầu bằng **phí đặt sân/địa điểm tiện lợi hoặc hoa hồng đặt sân**, sau đó thử phí tham gia trận/nhóm chơi. Không nên mở đầu bằng subscription premium rộng vì WTP chắc chắn còn thấp và phụ thuộc mật độ người dùng.

## Go-To-Market
- Launch theo cụm nhỏ: 1-2 trường đại học hoặc khu dân cư có sân cầu lông/bóng đá/bóng chuyền gần nhau.
- Chọn một hoặc hai môn có nhu cầu lặp lại cao để tạo liquidity trước.
- Kết hợp cộng đồng sinh viên, nhóm Zalo/Messenger hiện hữu, và đối tác sân để kéo cả hai phía marketplace.

## Risks / Weak Signals
- Mẫu không đại diện: nhóm 18-22 và học sinh/sinh viên áp đảo.
- WTP chắc chắn thấp: chỉ {metrics['wtp_strong']}/{N} trả lời "Có".
- Không có câu hỏi giá cụ thể nên chưa định lượng được price sensitivity.
- Không có dữ liệu hành vi thật như click, booking, attendance, retention.
- Hạn chế phương pháp: Thiếu thông tin tổng thể để tính toán response rate; khảo sát trực tuyến tự nguyện làm tăng rủi ro lệch mẫu.

## Next Validation Steps
- Concierge MVP: tạo nhóm ghép trận thủ công theo một môn/khu vực trong 2-4 tuần.
- Landing page test: đo conversion đăng ký, chọn môn, chọn lịch, sẵn sàng đặt cọc.
- Booking pilot: hợp tác vài sân để test lịch trống, giá, và phí tiện ích.
- WTP test: thử mức phí cụ thể thay vì chỉ hỏi ý định trả phí chung.
- Survey bổ sung: lấy mẫu người đi làm, người đặt sân thường xuyên, chủ sân, và nhóm tuổi ngoài sinh viên.

## Final Decision
**{final_decision}** Nên tiếp tục MVP, nhưng phạm vi nên hẹp: teammate matching + trust + court availability/price cho một nhóm early adopter cụ thể. Chưa nên tuyên bố product-market fit hoặc demand đại diện toàn TP.HCM.
"""
(REPORTS_DIR / "startup_survey_insights.md").write_text(insights_md, encoding="utf-8")

methodology_md = f"""# Báo cáo phương pháp khảo sát và tuân thủ

## 5.2.1 Objective & Target Audience
### 5.2.1.1 Mục tiêu khảo sát
Mục tiêu là đánh giá nhu cầu với ứng dụng tìm bạn chơi thể thao, ghép người theo khu vực/trình độ/lịch rảnh, hỗ trợ đặt sân hoặc tìm địa điểm, và kiểm tra tín hiệu sẵn sàng trả phí.

Các giả thuyết phân tích:
{hypothesis_rows}

### 5.2.1.2 Target audience và sample
- Target audience suy luận: người đang chơi, từng chơi, hoặc muốn bắt đầu chơi thể thao tại TP.HCM, đặc biệt nhóm cần tìm người chơi cùng hoặc cần đặt sân.
- Sample thực tế trong CSV: {N} phản hồi.
- Mẫu bị lệch: {int(clean['age'].eq('18–22').sum())}/{N} ở độ tuổi 18-22; {int(clean['occupation'].eq('Học sinh/Sinh viên').sum())}/{N} là học sinh/sinh viên.
- Kết luận tuân thủ: có thể dùng để phân tích nhóm trẻ/sinh viên, nhưng không đủ căn cứ để đại diện cho toàn bộ thị trường TP.HCM.

![Phân bố độ tuổi](charts/age_distribution.png)
![Phân bố nghề nghiệp](charts/occupation_distribution.png)

## 5.2.2 Survey Method
- Phương pháp khảo sát: Khảo sát trực tuyến (online survey) được thực hiện thông qua công cụ Google Forms.
- Phù hợp mục tiêu: Google Forms phù hợp với đối tượng mục tiêu trẻ tuổi (học sinh/sinh viên), giúp tối ưu hóa chi phí và nguồn lực hạn chế của startup trong giai đoạn khảo sát ý tưởng. Tuy nhiên, phương thức phân phối online qua mạng xã hội và nhóm chat dẫn đến rủi ro mẫu thuận tiện (convenience sampling), làm tăng độ lệch của mẫu nhân khẩu học.

## 5.2.3 Questionnaire Design
- Số câu hỏi phân tích: 21 câu nội dung cộng UID định danh kỹ thuật.
- Loại câu hỏi có trong khảo sát: single choice, multiple choice, Likert/ordinal scale, và open-ended response.
- Ví dụ single choice: tuổi, giới tính, nghề nghiệp, thu nhập, mức quan tâm thể thao, tần suất lý tưởng, WTP.
- Ví dụ multiple choice: môn thể thao, pain points, trust factors, cách tìm người, cách đặt sân, switching triggers, paid features.
- Ví dụ Likert/ordinal: mức sẵn sàng chơi với người mới từ 1 đến 5.
- Ví dụ open-ended: điều gì khiến respondent quan tâm hơn đến việc chơi thể thao.
- Đánh giá bias: câu hỏi số 18 và 19 mô tả app theo hướng có lợi nên có thể tạo hypothetical bias; báo cáo business không được xem các câu này là bằng chứng hành vi thật.

## 5.2.4 Recruitment
- Phương thức tuyển mẫu: Người trả lời được tiếp cận trực tuyến thông qua việc chia sẻ liên kết khảo sát Google Forms trên các kênh mạng xã hội, hội nhóm thể thao và cộng đồng sinh viên.
- Tuân thủ pháp lý & Contact Sourcing: Khảo sát thu thập dữ liệu bằng cách để người tham gia tự nguyện nhấp vào liên kết và điền thông tin trực tiếp, không sử dụng danh sách liên hệ từ bên thứ ba (no third-party contact list was used), đảm bảo tuân thủ các quy định về quyền riêng tư và điều khoản sử dụng của nền tảng.

## 5.2.5 Incentives and Compensation
- Chính sách bồi thường/khuyến khích: Khảo sát hoàn toàn không sử dụng bất kỳ phần thưởng, quà tặng hay lợi ích tài chính nào để khuyến khích người tham gia trả lời (no incentives).
- Tác động đến dữ liệu: Việc không cung cấp incentive giúp loại bỏ hoàn toàn rủi ro sai lệch do phần thưởng (response bias / incentive bias) - nơi người dùng trả lời giả tạo hoặc hời hợt chỉ để nhận quà. Toàn bộ phản hồi phản ánh động lực và mối quan tâm thực sự của đối tượng tham gia.

## 5.2.6 Anonymity
- Bảo vệ danh tính: Danh tính cá nhân của người tham gia khảo sát được bảo mật và ẩn danh hoàn toàn (personal identity is hidden). Tên của người tham gia không được thu thập.
- Thu thập và xử lý email (Yêu cầu 5.2.6.2): Để tuân thủ yêu cầu môn học về việc thu thập email nhưng vẫn bảo mật danh tính, địa chỉ email của người phản hồi được thu thập thông qua Google Forms nhưng được tách riêng biệt ngay từ đầu. Dữ liệu phân tích trong `response_details.csv` chỉ chứa mã định danh duy nhất `UID` đã được ẩn danh hóa. File lưu trữ địa chỉ email (PII) được cất giữ ở một vị trí bảo mật khác và chỉ được liên kết nội bộ khi cần thiết, đảm bảo không có bất kỳ thông tin nhận dạng cá nhân nào bị lộ trong các báo cáo và phân tích công khai.

## 5.2.7 Tools
- Công cụ thu thập dữ liệu: Google Forms.
- Công cụ phân tích và trực quan hóa dữ liệu: Python, Jupyter Notebook, thư viện pandas, numpy và matplotlib.
- Notebook thực thi chính: `survey_analysis.ipynb`.

## 5.2.8 Data Retention
- Chính sách lưu trữ dữ liệu: Dữ liệu ẩn danh (gắn với `UID`) được lưu trữ để phục vụ cho phân tích và cải tiến sản phẩm. Đối với file thông tin nhận dạng cá nhân (PII) chứa địa chỉ email, dữ liệu này được lưu trữ riêng biệt tại một thư mục an toàn và sẽ được xóa bỏ/hủy hoàn toàn sau thời hạn nghiên cứu là 6 tháng. Báo cáo công khai chỉ sử dụng bộ dữ liệu đã ẩn danh hóa.

## 5.2.9 Informed Consent
- Tự nguyện tham gia: Khảo sát được thực hiện trên tinh thần hoàn toàn tự nguyện, không có bất kỳ hình thức ép buộc hay ràng buộc nào đối với người trả lời.
- Cơ chế chấp thuận (Consent): Form khảo sát được thiết kế với phần giới thiệu rõ ràng về mục đích nghiên cứu. Người tham gia tự nguyện hoàn thành và gửi biểu mẫu Google Forms, hành động này được coi là sự đồng ý (implied consent) cho phép nhóm sử dụng dữ liệu phản hồi đã ẩn danh hóa phục vụ mục đích nghiên cứu môn học và phát triển sản phẩm. Cam kết thông tin cá nhân (email) không bị công bố công khai và sẽ được xóa sạch sau 6 tháng.

## 5.2.10 Sampling, Response Rate and Selection
- Response rate: không tính được vì không có số người được mời.
- Representativeness: không thể tuyên bố đại diện thị trường rộng vì mẫu lệch sinh viên/18-22 và recruitment database không được chứng minh.
- Cách diễn giải đúng: kết quả phản ánh sample hiện tại, hữu ích cho giả thuyết early adopter, không phải market sizing toàn TP.HCM.

## 5.2.11 Analysis
- Closed questions được dùng để tính count, percentage, segment comparison và hypothesis evidence.
- Single-value statistics không đứng một mình; báo cáo luôn so sánh theo giả thuyết hoặc segment.
- Ví dụ: WTP không chỉ nêu tỷ lệ “Có”, mà so với conditional WTP, pain frequency, booking need và paid feature preference.
- Bảng hỗ trợ: `question_summaries.csv`, `segment_comparison.csv`, `hypotheses.csv`, `feature_priority.csv`, `wtp_summary.csv`.

## 5.2.12 Conclusions Based on Survey Result
### Demographic conclusion
Respondents chủ yếu là 18-22 và học sinh/sinh viên. Điều này giúp chọn beachhead segment trong nhóm trẻ, nhưng giới hạn khả năng suy rộng sang người đi làm hoặc toàn bộ thị trường thể thao TP.HCM.

### Customer problems
Khách hàng gặp vấn đề trong tìm người chơi cùng, khớp lịch/trình độ, thiếu thông tin tin cậy khi chơi với người lạ, và đặt sân thủ công. Bằng chứng: `pain_points.csv`, `current_finding_pains.csv`, `booking_pains.csv`.

![Khó khăn khi tìm người chơi](charts/teammate_pain_points.png)
![Khó khăn khi đặt sân](charts/booking_pains.png)

### Customer interest
Có interest vì {metrics['frequency_lift_yes_or_maybe']}/{N} cho rằng app có thể hoặc chắc chắn giúp họ chơi thường xuyên hơn, và {metrics['openness_4_5']}/{N} sẵn sàng chơi với người mới ở mức 4-5. Tuy nhiên đây là stated interest, chưa phải hành vi thật.

![Tần suất thiếu người chơi](charts/teammate_shortage.png)
![Các yếu tố chuyển dịch người dùng sang app](charts/switching_triggers.png)

### Willingness to pay
WTP có tín hiệu nhưng chưa mạnh: {metrics['wtp_any']}/{N} có tín hiệu trả phí nếu tính cả câu trả lời có điều kiện, nhưng chỉ {metrics['wtp_strong']}/{N} trả lời chắc chắn “Có”. Nên ưu tiên phí gắn với giao dịch rõ ràng như đặt sân/địa điểm hoặc tham gia trận.

![Khảo sát độ sẵn sàng trả phí](charts/wtp_summary.png)

### Other relevant conclusion
Cần validation tiếp theo bằng MVP hẹp, đo conversion thực tế, booking thực tế, retention, và price test. Survey hiện tại đủ để tạo hypothesis và MVP direction, chưa đủ để chứng minh product-market fit.

#### Các hiểu biết ẩn sâu (Hidden Insights)
{hidden_rows_h4}
"""

(REPORTS_DIR / "survey_methodology_report.md").write_text(methodology_md, encoding="utf-8")

readme_md = """# Sports Survey Analysis

This project analyzes `response_details.csv` for a sports teammate-finding and court-booking app concept in Ho Chi Minh City.

## Setup

```bash
uv venv .venv
uv pip install --python .venv/bin/python pandas numpy matplotlib nbformat nbclient ipykernel notebook pypandoc-binary
```

## Run the notebook

```bash
.venv/bin/python -m ipykernel install --user --name exe-data-analytic --display-name "Python (exe-data-analytic)"
.venv/bin/jupyter notebook survey_analysis.ipynb
```

Run all cells from top to bottom. The notebook preserves `response_details.csv` and writes derived outputs only.

## Generated outputs

- `data/cleaned_survey.csv`
- `reports/startup_survey_insights.md`
- `reports/survey_methodology_report.md`
- `reports/startup_survey_insights.docx`
- `reports/survey_methodology_report.docx`
- `reports/charts/*.png`
- `reports/tables/*.csv`

## Notes

- `response_details.csv` is treated as anonymized survey data keyed by `UID`.
- The analysis does not load or expose the separate PII/email file.
- The survey was executed via Google Forms, with no incentives offered, and personal identity protected by separating the PII/email data.
"""
(ROOT / "README.md").write_text(readme_md, encoding="utf-8")

import os
try:
    import pypandoc
    # Convert md reports to docx
    old_cwd = os.getcwd()
    os.chdir(REPORTS_DIR)
    pypandoc.convert_file("startup_survey_insights.md", "docx", outputfile="startup_survey_insights.docx")
    pypandoc.convert_file("survey_methodology_report.md", "docx", outputfile="survey_methodology_report.docx")
    os.chdir(old_cwd)
    print("Successfully converted reports to DOCX.")
except Exception as e:
    print("Could not convert reports to DOCX using pypandoc:", e)

print("Generated outputs:")
for path in [DATA_DIR / "cleaned_survey.csv", REPORTS_DIR / "startup_survey_insights.md", REPORTS_DIR / "survey_methodology_report.md", REPORTS_DIR / "startup_survey_insights.docx", REPORTS_DIR / "survey_methodology_report.docx", ROOT / "README.md"]:
    print("-", path.relative_to(ROOT))
print(f"Tables: {len(list(TABLES_DIR.glob('*.csv')))}")
print(f"Charts: {len(list(CHARTS_DIR.glob('*.png')))}")


Successfully converted reports to DOCX.
Generated outputs:
- data/cleaned_survey.csv
- reports/startup_survey_insights.md
- reports/survey_methodology_report.md
- reports/startup_survey_insights.docx
- reports/survey_methodology_report.docx
- README.md
Tables: 20
Charts: 8
